# 25 — Async Execution

Run LangChain chains concurrently for better throughput.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import asyncio, time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Example 1: Sequential vs Concurrent

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = ChatPromptTemplate.from_template("What is {topic}? One sentence.") | llm | StrOutputParser()
topics = [{"topic": t} for t in ["photosynthesis", "the Doppler effect", "machine learning", "the Krebs cycle"]]

# Sequential
start = time.time()
for t in topics: chain.invoke(t)
seq_time = time.time() - start

# Concurrent
start = time.time()
results = await asyncio.gather(*[chain.ainvoke(t) for t in topics])
async_time = time.time() - start

print(f"Sequential: {seq_time:.2f}s\nConcurrent: {async_time:.2f}s\nSpeedup: {seq_time/async_time:.1f}x\n")
for t, r in zip(topics, results):
    print(f"  {t['topic']}: {r[:80]}...")

## Example 2: Async Batch Processing

In [ ]:
translate = ChatPromptTemplate.from_template("Translate to {language}: {text}") | llm | StrOutputParser()
items = [{"text": "Hello, how are you?", "language": lang} for lang in ["French", "Spanish", "Japanese", "German", "Portuguese"]]

start = time.time()
results = await asyncio.gather(*[translate.ainvoke(item) for item in items])
elapsed = time.time() - start

for item, result in zip(items, results):
    print(f"  {item['language']}: {result}")
print(f"\n  {len(items)} translations in {elapsed:.2f}s")

## Example 3: Async Streaming

In [ ]:
chain = ChatPromptTemplate.from_template("Write a haiku about {subject}.") | llm | StrOutputParser()

async for chunk in chain.astream({"subject": "programming"}):
    print(chunk, end="", flush=True)
print()

## Example 4: Multi-Chain Concurrent Execution

In [ ]:
summary_chain = ChatPromptTemplate.from_template("Summarise in one sentence: {text}") | llm | StrOutputParser()
keyword_chain = ChatPromptTemplate.from_template("Extract 3 keywords from: {text}") | llm | StrOutputParser()
sentiment_chain = ChatPromptTemplate.from_template("Sentiment (positive/negative/neutral) of: {text}") | llm | StrOutputParser()

text = "LangChain has become one of the most popular frameworks for building LLM applications. Its modular design and excellent documentation make it accessible to developers of all levels."

start = time.time()
summary, keywords, sentiment = await asyncio.gather(
    summary_chain.ainvoke({"text": text}),
    keyword_chain.ainvoke({"text": text}),
    sentiment_chain.ainvoke({"text": text}),
)
print(f"Summary: {summary}\nKeywords: {keywords}\nSentiment: {sentiment}\nCompleted in {time.time()-start:.2f}s")